## Before running

A virtual environment can be created using 
- 'pipenv install'
- 'pipenv shell'

This will allow us to all use the same packages and versions. They are listed in the Pipfile

In [ ]:
from refactoring import *

## Inputs

Dictionaries are taken as input from a parameter file, they contain the parameters for each soap descriptor

In [ ]:
descDict1 = {'lower': 1, 'upper': 50, 'centres': '{8, 7, 6, 1, 16, 17, 9}',
             'neighbours': '{8, 7, 6, 1, 16, 17, 9}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 1, 'max_cutoff': 50, 'min_sigma': 0.1, 
             'max_sigma': 0.9}

descDict2 = {'lower': 51, 'upper': 100, 'centres': '{8, 7, 6, 1, 16, 17, 9}',
             'neighbours': '{8, 7, 6, 1, 16, 17, 9}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50,
             'min_cutoff': 51, 'max_cutoff': 100, 'min_sigma': 1.1, 
             'max_sigma': 1.9}

Other parameters are also taken as input. These are automatically checked that the parameters are viable

In [ ]:
num_gens = 100
best_sample, lucky_few, population_size, number_of_children = 4, 2, 12, 4
early_stop = 2
early_number = 3 
min_generations = 5

## GeneParameter

GeneParameter class is created from each descriptor dictionary. 

In [ ]:
params1 = GeneParameters(**descDict1)
params2 = GeneParameters(**descDict2)

In [ ]:
params1

## GeneSet

We can use these classes to create a specific set of parameters that are consistant with these values. This returns a randomly generated GeneSet class

In [ ]:
example_gene_set = params1.make_gene_set()
example_gene_set

We can get the parameters used to create the GeneSet class

In [ ]:
example_gene_set.gene_parameters

We can get a descriptor string to be used as an input for getting SOAPs

In [ ]:
example_gene_set.get_soap_string()

We can also mutate the gene using the mutation chance in the GeneParameters class

In [ ]:
print(f"Before mutation {example_gene_set}")
example_gene_set.mutate_gene()
print(f"After mutation {example_gene_set}")

## Individual

An Individual is made up of a list of GeneSet classes.

In [ ]:
example_gene_set_two = params2.make_gene_set()
gene_set_list = [example_gene_set, example_gene_set_two]
example_individual = Individual(gene_set_list)
example_individual

Getting the score for an indivudual

In [ ]:
example_individual.get_score()
example_individual.score

Breeding two individuals to create a child. Mutation is automatically performed during this

In [ ]:
example_individual_two = Individual(gene_set_list)
print(f"Breeding {example_individual} with {example_individual_two}")
child = breed_individuals(example_individual, example_individual_two)
print(f"Created child {child}")

## Population

A Population is a collection of Individual classes. This can be created using a list of GeneParameter classes

In [ ]:
gene_parameters = [params1, params2]
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)
pop

To initialise the population

In [ ]:
pop.initialise_population()

If you want a way of neatly seeing what individuals are in the population

In [ ]:
pop.print_population()

The next generation can then be generated 

In [ ]:
pop.next_generation()
pop.print_population()

So to run the full GA 

In [ ]:
for _ in range(num_gens):
    pop.next_generation()
pop.print_population()

## BestHistory

BestHistory is a class to store the history and check convergence criteria. So the entire GA can be run, printed, and saved using the following code snippet:

In [ ]:
hist = BestHistory(early_stop, early_number, min_generations)
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)

pop.initialise_population()    
for gen in range(num_gens):
    if hist.converged:
        break
    print(f"Generation {gen}")
    pop.next_generation()
    hist.append(pop)
    print("-------")

There now exists the entire history of the best Individuals throughout each generation that can be saved and easily accessed. 

In [ ]:
vars(hist)

In [1]:
from refactoring import *
descDict1 = {'lower': 1, 'upper': 50, 'centres': '{8, 7, 6, 1, 16, 17, 9}',
             'neighbours': '{8, 7, 6, 1, 16, 17, 9}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 1, 'max_cutoff': 50, 'min_sigma': 0.1, 
             'max_sigma': 0.9}
params1 = GeneParameters(**descDict1)
example_gene_set = params1.make_gene_set()

conf_s, data = get_conf()

example_gene_set.cutoff = 5
example_gene_set.l_max = 3
example_gene_set.n_max = 6
test_ind = Individual([example_gene_set])

2022-11-04 11:29:00.269102: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


conf_s file exists


In [ ]:
# test_ind.comp_soaps(conf_s, data)

In [ ]:
test_ind.soaps.shape

In [ ]:
def cv_split(individual, splits, repeats, random_state = 999, regression = True):
    """
    Returns split indices for train and test sets
    """
    cv = RepeatedKFold(n_splits = splits, n_repeats = repeats, random_state = random_state)
    if regression:
        for train_index, test_index in cv.split(individual.soaps):
            res = get_scores_regression(individual, train_index, test_index)
#             print(scores)
    else:
        encoder = LabelEncoder()
        print(individual.targets)
        individual.targets = encoder.fit_transform(individual.targets)
        print(individual.targets)
        for train_index, test_index in cv.split(individual.soaps):
            res = get_scores_classification(individual, train_index, test_index, encoder = encoder)
#             print(scores)

In [ ]:
def scaleData(train, test):
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train)
    test_scaled = scaler.transform(test)

    return train_scaled, test_scaled, scaler

In [ ]:
def build_model(X):
    backend.clear_session()
    input_layer = Input(X.shape[1])
    hidden_layer = input_layer
    for layer in [30,30,30]:
        hidden_layer = Dense(layer, activation='relu')(hidden_layer)

    output_layer = Dense(units=1, activation='linear')(hidden_layer)
    model = Model(input_layer, output_layer)
    model.compile(loss='mean_squared_error', optimizer=optimizers.Adam(learning_rate=0.01), metrics=['mean_squared_error'])

    return model

In [ ]:
def get_scores_regression(individual, train_index, test_index, **kwargs):
    estimator = buildModel(individual.soaps)
    X_train, X_test, X_scaler = scaleData(individual.soaps[train_index], individual.soaps[test_index])
    y_train, y_test, y_scaler = scaleData(individual.targets[train_index].reshape(-1,1), individual.targets[test_index].reshape(-1,1))
    res = scorer_NN_regression(estimator, X_train, X_test, y_train, y_test, y_scaler)
    individual(add_to_results_dictionary(res))
    return scores

def get_scores_classification(individual, train_index, test_index, **kwargs):
    estimator = buildModel(individual.soaps)
    X_train, X_test, X_scaler = scaleData(individual.soaps[train_index], individual.soaps[test_index])
    encoder.transform(Y)
    scores = scorer_NN_class(estimator, X_train, X_test, y_train, y_test, y_scaler)
    return scores

In [ ]:
def scorer_NN_regression(estimator, X_train, X_test, y_train, y_test, y_scaler):
    """ Scoring function for use with NN regressor. Added by Matt. """

    callback = EarlyStopping(monitor='val_loss', patience=50)
    estimator.fit(X_train, y_train, callbacks=[callback], validation_split=0.1, epochs=200, verbose=False)
    y_test_pred, y_train_pred = estimator.predict(X_test, verbose=False), estimator.predict(X_train, verbose=False)
    y_test_pred, y_train_pred = y_scaler.inverse_transform(y_test_pred), y_scaler.inverse_transform(y_train_pred)
    y_test = np.ravel(y_test)
    y_train = np.ravel(y_train)
    testCorr = pearsonr(y_test, y_test_pred)[0]
    trainCorr = pearsonr(y_train, y_train_pred)[0]
    testMSE = mean_squared_error(y_test, y_test_pred)
    trainMSE =  mean_squared_error(y_train, y_train_pred)
    score = (2 * (trainMSE * (1-trainCorr)) + (testMSE * (1-testCorr)))
    res = [('scores', score), ('y_train_actual', y_train), ('y_test_actual', y_test), ('y_train_predictions', y_train_pred), ('y_test_predictions', y_test_pred)]
    return res

def scorer_NN_class(estimator, X_train, X_test, y_train, y_test, y_scaler):
    """ Scoring function for use with NN classifier. Added by Trent. """
    estimator.fit(X_train, y_train, callbacks=[callback], validation_split=0.1, epochs=200, verbose=False)
    y_test_pred, y_train_pred = estimator.predict(X_test, verbose=False), estimator.predict(X_train, verbose=False)
    y_test_pred, y_train_pred = y_scaler.inverse_transform(y_test_pred), y_scaler.inverse_transform(y_train_pred)
    y_test = np.ravel(y_test)
    y_train = np.ravel(y_train)
    testCorr = pearsonr(y_test, y_test_pred)[0]
    trainCorr = pearsonr(y_train, y_train_pred)[0]
    testMSE = mean_squared_error(y_test, y_test_pred)
    trainMSE =  mean_squared_error(y_train, y_train_pred)
    return (2 * (trainMSE * (1-trainCorr)) + (testMSE * (1-testCorr))), X_train, X_test, y_train, y_test, y_test_pred, y_train_pred

In [2]:
test_ind.get_score()

conf_s file exists
Getting soap for Atoms(symbols='COC6NCOC4ClC7O2C2O2H18', pbc=False)
Getting soap for Atoms(symbols='C2ONC6OH9', pbc=False)
Getting soap for Atoms(symbols='C3SO2C6FCONC7NCF3OH14', pbc=False)
Getting soap for Atoms(symbols='C6NC6NC3OC7NOH25', pbc=False)
Getting soap for Atoms(symbols='C25H34O6', pbc=False)
Getting soap for Atoms(symbols='C3SCONC5O2H15', pbc=False)
Getting soap for Atoms(symbols='COC6OC2NC3OC12NOH26', pbc=False)
Getting soap for Atoms(symbols='C10N2C6SO2NCF3H14', pbc=False)
Getting soap for Atoms(symbols='C9ONCOCCl2ONO2H12', pbc=False)
Getting soap for Atoms(symbols='C2NC2NC22H28', pbc=False)
Getting soap for Atoms(symbols='C14ClOC6NCH26', pbc=False)
Getting soap for Atoms(symbols='C19ClNC2NCH17', pbc=False)
Getting soap for Atoms(symbols='C12OC10NOH27', pbc=False)
Getting soap for Atoms(symbols='C2OCSCONC3NCONFH10', pbc=False)
Getting soap for Atoms(symbols='C10OC8OH24', pbc=False)
Getting soap for Atoms(symbols='C9ONC6FC9FO2H21', pbc=False)
Getting so

TypeError: 'NoneType' object is not iterable

In [ ]:
cv_split(test_ind, 5, 1, random_state = 999, regression = True)

In [8]:
test_ind.results_dictionary['y_train_actual']

[array([0.37765957, 0.50531915, 0.68085106, 0.74468085, 0.2606383 ,
        0.46276596, 0.54787234, 0.40425532, 0.27659574, 0.39893617,
        0.65957447, 0.61702128, 0.69148936, 0.58510638, 0.4787234 ,
        0.22340426, 0.69680851, 0.7712766 , 0.        , 0.46808511,
        0.22340426, 0.7712766 , 0.81914894, 0.4893617 , 0.36170213,
        0.30851064, 0.56914894, 0.43085106, 0.63829787, 0.18617021,
        1.        , 0.20212766, 0.36702128, 0.62234043, 0.5       ,
        0.57978723, 0.56382979, 0.70744681, 0.79787234, 0.50531915,
        0.5106383 , 0.22340426, 0.25      , 0.62234043, 0.30851064,
        0.38829787, 0.19148936, 0.5106383 , 0.22340426, 0.31382979,
        0.50531915, 0.09042553, 0.34574468, 0.03723404, 0.80319149,
        0.32446809, 0.12765957, 0.7393617 , 0.5106383 , 0.04255319,
        0.29787234, 0.73404255, 0.63297872, 0.54787234, 0.43617021,
        0.37765957, 0.32978723, 0.21808511, 0.        , 0.5       ,
        0.43085106, 0.41489362, 0.38297872, 0.29

In [7]:
test_ind.results_dictionary

defaultdict(list,
            {'scores': [array([101602.67819791306], dtype=object)],
             'y_train_actual': [array([0.37765957, 0.50531915, 0.68085106, 0.74468085, 0.2606383 ,
                     0.46276596, 0.54787234, 0.40425532, 0.27659574, 0.39893617,
                     0.65957447, 0.61702128, 0.69148936, 0.58510638, 0.4787234 ,
                     0.22340426, 0.69680851, 0.7712766 , 0.        , 0.46808511,
                     0.22340426, 0.7712766 , 0.81914894, 0.4893617 , 0.36170213,
                     0.30851064, 0.56914894, 0.43085106, 0.63829787, 0.18617021,
                     1.        , 0.20212766, 0.36702128, 0.62234043, 0.5       ,
                     0.57978723, 0.56382979, 0.70744681, 0.79787234, 0.50531915,
                     0.5106383 , 0.22340426, 0.25      , 0.62234043, 0.30851064,
                     0.38829787, 0.19148936, 0.5106383 , 0.22340426, 0.31382979,
                     0.50531915, 0.09042553, 0.34574468, 0.03723404, 0.80319149,
     